Задача

In [ ]:
import math
n = int(input())
x = []
y = []

for i in range(n):
    a, b = map(int, input().split())
    x.append(a)
    y.append(b)

INF = 10**10
min_d = [INF] * n
used = [False] * n

min_d[0] = 0.0
total_weight = 0.0

for i in range(n):
    best_idx = -1
    best_val = INF
    
    for i in range(n):
        if not used[i] and min_d[i] < best_val:
            best_val = min_d[i]
            best_idx = i
    
    if best_idx == -1:
        break
    
    used[best_idx] = True
    total_weight += best_val
    
    x_new = x[best_idx]
    y_new = y[best_idx]
    
    for j in range(n):
        if not used[j]:
            dx = x_new - x[j]
            dy = y_new - y[j]
            dist = math.sqrt(dx*dx + dy*dy)
            
            if dist < min_d[j]:
                min_d[j] = dist

print(total_weight)

 3
 0 0
 1 0
 0 1


2.0


Задача 1

In [5]:
n = int(input())
dicti = set(input().split())
m, l = map(int, input().split())

board = []
for i in range(m):
    row_str = input()
    clean_chars = [char for char in row_str if char.isalpha()]
    board.append(clean_chars[:l])

found_words = set()

direct = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1),  (1, 0),  (1, 1)]

def dfs(r, c, current_word, visited):
    current_word += board[r][c]

    has_prefix = False
    for word in dicti:
        if word.startswith(current_word):
            has_prefix = True
            break

    if not has_prefix:
        return

    if current_word in dicti:
        found_words.add(current_word)

    visited.add((r, c))

    for dr, dc in direct:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < l and (nr, nc) not in visited:
            dfs(nr, nc, current_word, visited)

    visited.remove((r, c))

for i in range(m):
    for j in range(l):
        dfs(i, j, "", set())

result = sorted(list(found_words))
print(" ".join(result))

 4
 GEEKS FOR QUIZ GO
 3 3
 G I Z
 U E K
 Q S E


GEEKS QUIZ


Задача 2


Алгоритм Прима

In [6]:
import heapq

def prim_mst(graph, start=0):
    """
    graph: список смежности graph[vertex] = [(neighbor, weight), ...]
    Возвращает: (общий_вес, список_рёбер_mst)
    """
    n = len(graph)
    in_mst = [False] * n
    mst_edges = []
    total_weight = 0
    # Куча содержит кортежи (weight, from_vertex, to_vertex)
    # Изначально добавляем все рёбра из стартовой вершины
    priority_queue = []

    # Функция для добавления рёбер вершины v в кучу
    def add_edges(v):
        in_mst[v] = True
        for to, w in graph[v]:
            if not in_mst[to]:
                heapq.heappush(priority_queue, (w, v, to))
    
    # Начинаем со стартовой вершины
    add_edges(start)
    
    while priority_queue and len(mst_edges) < n - 1:
        weight, u, v = heapq.heappop(priority_queue)
        # Если вершина v уже добавлена, пропускаем (она могла быть добавлена через другой путь)
        if in_mst[v]:
            continue
        
        # Добавляем ребро в MST
        mst_edges.append((u, v, weight))
        total_weight += weight
        # Добавляем все рёбра из новой вершины
        add_edges(v)
    
    return total_weight, mst_edges

# Пример использования для того же графа (построим список смежности)
if __name__ == "__main__":
    V = 6
    adj_list = [[] for _ in range(V)]
    # Веса рёбер (weight, u, v) - добавляем в обе стороны
    edges_data = [
        (7, 0, 1), (8, 0, 2), (11, 1, 2), (2, 1, 3), (6, 2, 3),
        (9, 2, 4), (11, 3, 4), (9, 3, 5), (10, 4, 5)
    ]
    for w, u, v in edges_data:
        adj_list[u].append((v, w))
        adj_list[v].append((u, w))
    
    total_cost, mst = prim_mst(adj_list, start=0) # Начнем с вершины A
    print(f"\nАлгоритм Прима: Вес MST = {total_cost}")
    print("Рёбра в MST:")
    for u, v, w in mst:
        print(f"{chr(65+u)} -- {chr(65+v)} : {w}")
    # Ожидаемый вес: 33


Алгоритм Прима: Вес MST = 33
Рёбра в MST:
A -- B : 7
B -- D : 2
D -- C : 6
C -- E : 9
D -- F : 9


Алгоритм Краскала

In [7]:
class DSU:
    """Система непересекающихся множеств (Union-Find)"""
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        # Сжатие путей
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        # Объединение по рангу
        x_root, y_root = self.find(x), self.find(y)
        if x_root == y_root:
            return False
        if self.rank[x_root] < self.rank[y_root]:
            self.parent[x_root] = y_root
        elif self.rank[x_root] > self.rank[y_root]:
            self.parent[y_root] = x_root
        else:
            self.parent[y_root] = x_root
            self.rank[x_root] += 1
        return True

def kruskal_mst(vertices_count, edges):
    """
    edges: список кортежей (weight, u, v)
    Возвращает: (общий_вес, список_рёбер_mst)
    """
    # Сортируем рёбра по весу
    sorted_edges = sorted(edges, key=lambda edge: edge[0])
    
    dsu = DSU(vertices_count)
    mst_edges = []
    total_weight = 0
    
    for weight, u, v in sorted_edges:
        # Если вершины в разных компонентах, добавляем ребро
        if dsu.union(u, v):
            mst_edges.append((u, v, weight))
            total_weight += weight
            
    return total_weight, mst_edges

# Пример использования
if __name__ == "__main__":
    # Граф из примера (вершины: A=0, B=1, C=2, D=3, E=4, F=5)
    V = 6
    # Веса рёбер (weight, u, v)
    graph_edges = [
        (7, 0, 1),   # A-B
        (8, 0, 2),   # A-C
        (11, 1, 2),  # B-C
        (2, 1, 3),   # B-D
        (6, 2, 3),   # C-D
        (9, 2, 4),   # C-E
        (11, 3, 4),  # D-E
        (9, 3, 5),   # D-F
        (10, 4, 5)   # E-F
    ]
    
    total_cost, mst = kruskal_mst(V, graph_edges)
    print(f"Алгоритм Краскала: Вес MST = {total_cost}")
    print("Рёбра в MST:")
    for u, v, w in mst:
        print(f"{chr(65+u)} -- {chr(65+v)} : {w}")
    # Ожидаемый вес: 33

Алгоритм Краскала: Вес MST = 33
Рёбра в MST:
B -- D : 2
C -- D : 6
A -- B : 7
C -- E : 9
D -- F : 9


Асимптотическая сложность алгоритма Краскала составляет O(E log E), так как основная операция — это сортировка всех ребер по весу. Сложность алгоритма Прима с использованием равна O(E log V), поскольку для каждого ребра выполняется операция в куче, размер которой зависит от количества вершин. Теоретически на очень плотных графах Прим работает быстрее, так как логарифм берется от числа вершин, а не от большего числа ребер. На разреженных графах алгоритм Краскала часто оказывается быстрее благодаря высокой оптимизации встроенной сортировки. 

Задача 3

In [8]:
n, m = map(int, input().split())

edges = []
for i in range(m):
    u, v, w = map(int, input().split())
    edges.append((u, v, w, i))

sorted_edges = sorted(edges, key=lambda x: x[2])

parent = list(range(n + 1))

def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    rx = find(x)
    ry = find(y)
    if rx != ry:
        parent[rx] = ry
        return True
    return False

mst_weight = 0
in_mst = [False] * m
adj = [[] for i in range(n + 1)]

for u, v, w, idx in sorted_edges:
    if union(u, v):
        mst_weight += w
        in_mst[idx] = True
        adj[u].append((v, w))
        adj[v].append((u, w))

def get_max_on_path(current, target, prev_node, current_max):
    if current == target:
        return current_max
    for neighbor, weight in adj[current]:
        if neighbor != prev_node:
            new_max = max(current_max, weight)
            res = get_max_on_path(neighbor, target, current, new_max)
            if res != -1:
                return res
    return -1

results = [0] * m

for u, v, w, idx in edges:
    if in_mst[idx]:
        results[idx] = mst_weight
    else:
        max_w = get_max_on_path(u, v, -1, 0)
        results[idx] = mst_weight + w - max_w

for ans in results:
    print(ans)

 5 7
 1 2 3
 1 3 1
 1 4 5
 2 3 2
 2 5 3
 3 4 2
 4 5 4


9
8
11
8
8
8
9


Задача 4

In [9]:
import heapq

data = list(map(int, input().split()))
n = data[0]
m = data[1]
centers = data[2:]

adj = [[] for i in range(n)]
for i in range(m):
    u, v, w = map(int, input().split())
    adj[u].append((v, w))
    adj[v].append((u, w))

dist = [-1] * n
pq = []

for c in centers:
    dist[c] = 0
    heapq.heappush(pq, (0, c))

while pq:
    d, u = heapq.heappop(pq)
    
    if d > dist[u] and dist[u] != -1:
        continue
    
    for v, w in adj[u]:
        if dist[v] == -1 or d + w < dist[v]:
            dist[v] = d + w
            heapq.heappush(pq, (dist[v], v))

total_sum = 0
for d in dist:
    if d != -1:
        total_sum += d

print(total_sum)

 4 3 0 1
 0 1 34
 2 1 7
 3 2 85


99
